# BirdCLEF 2026 — Advanced Experiments

This notebook covers Models 2–7 from the plan:
- **Model 2**: EfficientNet-B4 + multi-scale mel (3 channels)
- **Model 3**: Audio Spectrogram Transformer (AST)
- **Model 4**: PANNs CNN14
- **Model 5**: EfficientNet + GRU temporal head
- **Model 6**: Pseudo-labeling
- **Model 7**: Ensemble

Each section is self-contained. Run sections independently.

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from config import Config, ALL_CLASSES, NUM_CLASSES
from dataset import build_dataloaders
from augmentation import Augmentation, batch_mixup
from evaluate import evaluate
from models import build_model
from audio_utils import build_mel_transform, soundscape_windows

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

---
## Model 2: EfficientNet-B4 with Multi-scale Mel (3-channel)

Stack 64-mel, 128-mel, 256-mel spectrograms as 3 channels.
Captures both coarse spectral shape and fine harmonic structure.

In [ ]:
import torchaudio.transforms as T
from audio_utils import load_audio, crop_or_pad, waveform_to_logmel

def make_multiscale_mel(waveform: torch.Tensor, cfg: Config) -> torch.Tensor:
    """Returns (3, 128, frames) — three mel resolutions resized to same height."""
    scales = [64, 128, 256]
    mels = []
    for n_mels in scales:
        t = T.MelSpectrogram(
            sample_rate=cfg.sample_rate, n_fft=cfg.n_fft,
            hop_length=cfg.hop_length, n_mels=n_mels,
            f_min=cfg.f_min, f_max=cfg.f_max, power=2.0,
        )
        db = T.AmplitudeToDB('power', top_db=80.0)
        spec = db(t(waveform)).squeeze(0)  # (n_mels, frames)
        # Resize all to 128 mel bins
        spec = torch.nn.functional.interpolate(
            spec.unsqueeze(0).unsqueeze(0), size=(128, spec.shape[-1]),
            mode='bilinear', align_corners=False
        ).squeeze(0).squeeze(0)
        # Normalize
        spec = (spec - spec.mean()) / (spec.std() + 1e-6)
        mels.append(spec)
    return torch.stack(mels, dim=0)  # (3, 128, frames)

# Test shape
cfg = Config()
dummy = torch.randn(1, cfg.clip_samples)
ms = make_multiscale_mel(dummy, cfg)
print('Multi-scale mel shape:', ms.shape)  # expected (3, 128, ~500)

In [ ]:
# EfficientNet-B4 with 3-channel input
from models.efficientnet_mel import EfficientNetMel

model_b4 = EfficientNetMel(num_classes=NUM_CLASSES, model_name='efficientnet_b4',
                            pretrained=True, n_scales=3).to(device)
total = sum(p.numel() for p in model_b4.parameters())
print(f'EfficientNet-B4 3-scale params: {total:,}')

# Quick forward pass test
x = torch.randn(2, 3, 128, 500).to(device)
out = model_b4(x)
print('Output shape:', out.shape)  # (2, 234)
print('Model 2 ready — use train.py with --model efficientnet_b4 and a custom dataset wrapper')

---
## Model 3: Audio Spectrogram Transformer (AST)

Uses `MIT/ast-finetuned-audioset-10-10-0.4593` from HuggingFace.

**Install**: `pip install transformers`

In [ ]:
from models.ast_model import ASTModel

cfg_ast = Config()
cfg_ast.model_name      = 'ast'
cfg_ast.experiment_name = 'exp002_ast'
cfg_ast.learning_rate   = 3e-5   # lower LR for transformer fine-tuning
cfg_ast.epochs          = 20
cfg_ast.warmup_epochs   = 3

print('Loading AST model (downloads ~350MB on first run)...')
model_ast = ASTModel(num_classes=NUM_CLASSES, freeze_layers=8).to(device)
total = sum(p.numel() for p in model_ast.parameters())
trainable = sum(p.numel() for p in model_ast.parameters() if p.requires_grad)
print(f'Total: {total:,}  Trainable: {trainable:,}')

# Forward pass test
x = torch.randn(2, 1, 128, 500).to(device)
out = model_ast(x)
print('AST output shape:', out.shape)

In [ ]:
# Training: unfreeze all layers after epoch 5
# Run from terminal for a full training run:
# python src/train.py --model ast --experiment exp002_ast --lr 3e-5 --epochs 20

# Or demonstrate the unfreeze callback:
print('After epoch 5, call model_ast.unfreeze_all() to fine-tune the full transformer.')

---
## Model 4: PANNs CNN14

Re-implemented CNN14 architecture. Load pretrained AudioSet weights if available:
```bash
wget https://zenodo.org/record/3987831/files/Cnn14_mAP%3D0.431.pth -O checkpoints/panns_cnn14.pth
```

In [ ]:
from models.panns_cnn14 import PANNSCNN14

panns_weights = Path('../checkpoints/panns_cnn14.pth')

model_cnn14 = PANNSCNN14(
    num_classes=NUM_CLASSES,
    pretrained=panns_weights.exists(),
    weights_path=str(panns_weights) if panns_weights.exists() else None,
).to(device)

total = sum(p.numel() for p in model_cnn14.parameters())
print(f'CNN14 params: {total:,}')

x = torch.randn(2, 1, 128, 500).to(device)
print('CNN14 output:', model_cnn14(x).shape)

---
## Model 5: EfficientNet + GRU Temporal Head

In [ ]:
from models.temporal_rnn import TemporalRNN

model_rnn = TemporalRNN(num_classes=NUM_CLASSES, pretrained=True).to(device)
total = sum(p.numel() for p in model_rnn.parameters())
print(f'TemporalRNN params: {total:,}')

x = torch.randn(2, 1, 128, 500).to(device)
print('TemporalRNN output:', model_rnn(x).shape)

---
## Model 6: Pseudo-labeling

Steps:
1. Load a trained checkpoint (e.g. best EfficientNet-B3)
2. Run inference on all 10,657 soundscape files
3. Threshold at 0.85 → treat as pseudo ground truth
4. Retrain with extended dataset

In [ ]:
THRESHOLD = 0.85
CKPT_PATH = '../checkpoints/exp001_baseline/best.pt'

cfg_pl = Config()
mel = build_mel_transform(cfg_pl)

if not Path(CKPT_PATH).exists():
    print(f'Checkpoint not found at {CKPT_PATH}. Train the baseline first.')
else:
    ckpt = torch.load(CKPT_PATH, map_location=device)
    teacher = build_model('efficientnet_b3', NUM_CLASSES, pretrained=False)
    teacher.load_state_dict(ckpt['model'])
    teacher = teacher.to(device).eval()

    pseudo_rows = []
    sc_files = sorted(cfg_pl.train_soundscapes_dir.glob('*.ogg'))
    print(f'Running pseudo-labeling on {len(sc_files)} soundscapes...')

    with torch.no_grad():
        for path in sc_files[:50]:  # limit to 50 for demo — remove limit for full run
            windows = soundscape_windows(path, cfg_pl, mel)
            for end_sec, log_mel in windows:
                spec   = log_mel.unsqueeze(0).to(device)
                probs  = torch.sigmoid(teacher(spec)).squeeze(0).cpu()
                labels = [ALL_CLASSES[i] for i, p in enumerate(probs) if p >= THRESHOLD]
                if labels:
                    pseudo_rows.append({
                        'filename': path.name,
                        'start_sample': (end_sec - 5) * cfg_pl.sample_rate,
                        'labels': labels,
                    })

    print(f'Generated {len(pseudo_rows)} high-confidence pseudo-labeled segments')
    # These segments can be passed to SoundscapeDataset for retraining

---
## Model 7: Ensemble

Average predictions from EfficientNet-B3, AST, and CNN14.

In [ ]:
from predict import load_model_from_checkpoint, predict_soundscape_ensemble

CHECKPOINTS = {
    'efficientnet_b3': '../checkpoints/exp001_baseline/best.pt',
    'ast':             '../checkpoints/exp002_ast/best.pt',
    'panns':           '../checkpoints/exp003_panns/best.pt',
}

available = {name: path for name, path in CHECKPOINTS.items() if Path(path).exists()}
print(f'Available checkpoints: {list(available.keys())}')

if len(available) >= 2:
    models = [load_model_from_checkpoint(path, name, device) for name, path in available.items()]

    cfg_ens = Config()
    mel_ens = build_mel_transform(cfg_ens)

    test_files = sorted(cfg_ens.test_soundscapes_dir.glob('*.ogg'))
    if test_files:
        rows = predict_soundscape_ensemble(models, test_files[0], cfg_ens, mel_ens, device)
        print(f'Ensemble produced {len(rows)} rows for {test_files[0].name}')
    else:
        print('No test soundscapes found — run during competition.')
else:
    print('Train at least 2 models before ensembling. Available so far:', list(available.keys()))

In [ ]:
# Full ensemble submission (all test files):
# python src/predict.py --ensemble checkpoints/exp001_baseline/best.pt checkpoints/exp002_ast/best.pt
print('Run this once all models are trained:')
print('  python src/predict.py --ensemble \\')
print('    checkpoints/exp001_baseline/best.pt \\')
print('    checkpoints/exp002_ast/best.pt \\')
print('    checkpoints/exp003_panns/best.pt')